# Renewable Energy and CO2 Emissions in Latin America & the Caribbean

**Research question:** Does renewable energy adoption predict lower CO2 emissions per capita across Latin American and Caribbean (LAC) countries, controlling for economic growth (GDP per capita)?

**Why this question:** LAC has one of the world's cleanest electricity grids on average (hydropower especially), but emissions still vary a lot by country and are rising in some places as economies grow. Understanding the relationship between renewables and emissions -- while controlling for income -- is directly relevant to IDB's work on infrastructure and energy policy.

**Data source:** World Bank World Development Indicators (WDI), pulled via the `wbgapi` Python package.

**Structure of this notebook:**
1. Setup
2. Pull data (panel: country x year)
3. Clean and reshape
4. Exploratory plots
5. Regression (pooled OLS, then fixed effects)
6. Interpretation notes

> Run this in Google Colab or a local Jupyter environment with internet access -- it needs to reach the World Bank API.

## 1. Setup

If running in Colab, uncomment the `pip install` line below. `wbgapi` handles the World Bank API calls; `linearmodels` gives us panel fixed-effects regression (`statsmodels` alone doesn't do country+year fixed effects cleanly).

In [ ]:
# !pip install wbgapi linearmodels --quiet

import wbgapi as wb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
from linearmodels.panel import PanelOLS

pd.set_option('display.max_columns', None)

## 2. Pull data

**Indicators:**
- `EN.ATM.CO2E.PC` -- CO2 emissions (metric tons per capita) -- our outcome variable
- `EG.FEC.RNEW.ZS` -- renewable energy consumption (% of total final energy consumption) -- our variable of interest
- `NY.GDP.PCAP.KD` -- GDP per capita (constant 2015 US$) -- control
- `EG.ELC.ACCS.ZS` -- access to electricity (% of population) -- control

**Note:** The World Bank occasionally renames/retires indicator codes (especially CO2-related ones, since some moved to a Climate Watch source around 2024). If `EN.ATM.CO2E.PC` returns nothing, run `wb.series.info(q='CO2')` to find the current code and swap it in below.

In [ ]:
# Latin American & Caribbean countries (IDB member countries), by ISO3 code
lac_countries = [
    'ARG', 'BHS', 'BRB', 'BLZ', 'BOL', 'BRA', 'CHL', 'COL', 'CRI',
    'DOM', 'ECU', 'SLV', 'GTM', 'GUY', 'HTI', 'HND', 'JAM', 'MEX',
    'NIC', 'PAN', 'PRY', 'PER', 'SUR', 'TTO', 'URY', 'VEN'
]

indicators = {
    'EN.ATM.CO2E.PC': 'co2_per_capita',
    'EG.FEC.RNEW.ZS': 'renewable_pct',
    'NY.GDP.PCAP.KD': 'gdp_per_capita',
    'EG.ELC.ACCS.ZS': 'electricity_access_pct',
}

years = range(2000, 2023)

# quick sanity check: confirm the indicator codes still exist / see alternatives
# wb.series.info(q='CO2')

df = wb.data.DataFrame(
    list(indicators.keys()),
    economy=lac_countries,
    time=years,
    labels=True,
    skipBlanks=True,
)

df.head()

## 3. Clean and reshape

`wbgapi` returns data in a wide, multi-index format that's a bit awkward. We'll reshape it into a clean **long panel**: one row per (country, year), one column per indicator.

In [ ]:
# Reshape into long panel format
panel = wb.data.DataFrame(
    list(indicators.keys()),
    economy=lac_countries,
    time=years,
    skipBlanks=True,
).reset_index()

# wbgapi's long-format output columns are typically: economy, series, YR2000, YR2001, ...
# melt the year columns into rows
year_cols = [c for c in panel.columns if c.startswith('YR')]
panel_long = panel.melt(id_vars=['economy', 'series'], value_vars=year_cols,
                          var_name='year', value_name='value')
panel_long['year'] = panel_long['year'].str.replace('YR', '').astype(int)

# pivot so each indicator becomes its own column
panel_clean = panel_long.pivot_table(index=['economy', 'year'], columns='series', values='value').reset_index()
panel_clean = panel_clean.rename(columns=indicators)

print('Missing values per column:')
print(panel_clean.isna().sum())

# Drop rows missing our core variables (CO2, renewables, GDP)
panel_model = panel_clean.dropna(subset=['co2_per_capita', 'renewable_pct', 'gdp_per_capita']).copy()
panel_model['log_gdp_per_capita'] = np.log(panel_model['gdp_per_capita'])

panel_model.head()

## 4. Exploratory plots

Always look at the data before modeling it.

In [ ]:
# Renewable share over time, one line per country
fig, ax = plt.subplots(figsize=(10, 6))
for country, grp in panel_model.groupby('economy'):
    ax.plot(grp['year'], grp['renewable_pct'], alpha=0.6, label=country)
ax.set_title('Renewable Energy Share (% of final energy consumption), LAC 2000-2022')
ax.set_xlabel('Year')
ax.set_ylabel('Renewable share (%)')
plt.tight_layout()
plt.show()

In [ ]:
# Scatter: renewable share vs CO2 per capita
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(panel_model['renewable_pct'], panel_model['co2_per_capita'], alpha=0.4)
ax.set_xlabel('Renewable energy share (%)')
ax.set_ylabel('CO2 emissions per capita (metric tons)')
ax.set_title('Renewable Share vs CO2 per Capita (all country-years)')
plt.tight_layout()
plt.show()

## 5. Regression

**Step 1: Pooled OLS** (simplest baseline, ignores country/year structure)

In [ ]:
X = panel_model[['renewable_pct', 'log_gdp_per_capita']]
X = sm.add_constant(X)
y = panel_model['co2_per_capita']

pooled_model = sm.OLS(y, X).fit()
print(pooled_model.summary())

**Step 2: Fixed-effects panel regression**

This controls for each country's unique baseline level (e.g., Trinidad's oil economy vs. Costa Rica's) and for global year-to-year shocks (e.g., 2020 COVID dip). This is the more credible model -- it isolates the *within-country, over-time* relationship rather than just comparing countries to each other.

In [ ]:
panel_indexed = panel_model.set_index(['economy', 'year'])

y_fe = panel_indexed['co2_per_capita']
X_fe = panel_indexed[['renewable_pct', 'log_gdp_per_capita']]

fe_model = PanelOLS(y_fe, X_fe, entity_effects=True, time_effects=True).fit()
print(fe_model)

## 6. Interpretation notes (fill this in after running)

Answer these once you have real output:

- What is the sign and size of the `renewable_pct` coefficient in the fixed-effects model? Is it statistically significant (p < 0.05)?
- Does the result hold up between the pooled OLS and fixed-effects versions, or does it change? (If it changes a lot, that tells you cross-country differences were driving the pooled result.)
- **Caveats to state explicitly:** this is observational/correlational, not causal. Countries that adopt more renewables may differ in unobserved ways (political stability, geography favoring hydro, existing industrial base). A stronger causal design would need an instrument or a policy-based natural experiment (e.g., before/after a specific renewable energy subsidy).
- **Policy framing:** translate the finding into a one-paragraph takeaway written for a non-technical reader -- this is the actual IDB-style deliverable.